# Reasoning Scaling Laws

Training and benchmark pipeline testing H1 (data efficiency vs. model size) and H2 (agentic compute vs. model scale) on GSM8K.

**Run order:** Cell 0 (env vars) → Cell 1 (install, then **restart kernel**) → Cell 2 (verify imports) → Cell 3 (download from GitHub) → Cell 4 (mount Drive) → Cell 5 (GPU check) → Cell 6 (config) → then either the Test Run cell, or Train 3B / Train 7B → Verify → Benchmark → Results.

**Important:** after running the install cell you must restart the kernel (Runtime → Restart session) before continuing — numpy/torch have C extensions that cannot reload in a running kernel. Re-run Cell 0 after the restart so the env vars are set again.

The 3B and 7B training cells can be run in separate Colab sessions — checkpoints are saved to Drive after every epoch.

In [1]:
!rm -rf /content/unsloth_compiled_cache

In [ ]:
# Cell 0 — Environment setup. MUST run before any unsloth import.
import os

print("Env cell ran.")

In [2]:
# Cell 1 — Install dependencies
# 1. Avinstallera trasiga versioner först
!pip uninstall -y bitsandbytes unsloth unsloth_zoo vllm

# 2. Installera rätt version av bitsandbytes anpassad för CUDA 12
!pip install --no-cache-dir bitsandbytes --extra-index-url https://download.pytorch.org/whl/cu121

# 3. Installera Unsloth och dess vänner på rätt sätt
!pip install --no-cache-dir unsloth unsloth_zoo vllm datasets trl matplotlib

# >>> After this finishes, RESTART THE KERNEL, then re-run Cell 0, then run Cell 2. <<<


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 164.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 149.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 266.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of vllm to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 318.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 MB 241.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869

In [1]:
import unsloth, trl, transformers, torch
import numpy as np, datasets, vllm

print("unsloth", unsloth.__version__)
print("trl", trl.__version__)
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("numpy", np.__version__)
print("dataset", datasets.__version__)
print("vllm",vllm.__version__)
print("CUDA tillgänglig:", torch.cuda.is_available())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.5.8
trl 0.24.0
transformers 4.57.6
torch 2.10.0+cu128
numpy 2.0.2
dataset 4.3.0
vllm 0.19.1
CUDA tillgänglig: True


In [2]:
%%bash
# Replace your-username and your-repo-name with your actual GitHub details
BRANCH="runBranch"
USER="huineith"
REPO="AgenticAndRL_LocalMathModels"

# Define the files to download
FILES=("utils.py" "prompts.py" "rewards.py" "grpo_trainer.py" "agent.py" "benchmarker.py" "warmup_data.json" "start_grpo_sub_process.py")

echo "Fetching fresh files from GitHub and overwriting local copies..."
for file in "${FILES[@]}"; do
    wget -q -O "/content/$file" "https://raw.githubusercontent.com/$USER/$REPO/$BRANCH/$file"
    echo " ✓ Updated: $file"
done

Fetching fresh files from GitHub and overwriting local copies...
 ✓ Updated: utils.py
 ✓ Updated: prompts.py
 ✓ Updated: rewards.py
 ✓ Updated: grpo_trainer.py
 ✓ Updated: agent.py
 ✓ Updated: benchmarker.py
 ✓ Updated: warmup_data.json
 ✓ Updated: start_grpo_sub_process.py


In [3]:
# Cell 2 — Mount Google Drive (needed for checkpoint and result saving)
import os
import sys
from google.colab import drive

drive.mount('/content/drive')

CONTENT_DIR = '/content'
os.chdir(CONTENT_DIR)
if CONTENT_DIR not in sys.path:
    sys.path.insert(0, CONTENT_DIR)

print(f'Working directory: {os.getcwd()}')
print('Source files were downloaded from GitHub in the previous cell.')

Mounted at /content/drive
Working directory: /content
Source files were downloaded from GitHub in the previous cell.


In [4]:
# Cell 3 — GPU diagnostics
import torch

if torch.cuda.is_available():
    name  = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free  = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
    print(f'GPU   : {name}')
    print(f'VRAM  : {total:.1f} GB total  |  {free:.1f} GB free')
    print(f'BF16  : {torch.cuda.is_bf16_supported()}')
else:
    print('WARNING: No GPU detected. Switch runtime to L4 GPU in Runtime > Change runtime type.')

GPU   : NVIDIA A100-SXM4-40GB
VRAM  : 42.4 GB total  |  42.4 GB free
BF16  : True


In [5]:
# Cell 4 — Configuration
MODEL_3B        = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MODEL_1_5B        = 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit'
DATA_FRACTIONS  = [0.025, 0.05, 0.10]
WARMUP_PATH     = 'warmup_data.json'
SEED            = 42
MAX_EPOCHS      = 3

DRIVE_SAVE_DIR  = '/content/drive/MyDrive/ExamensArbete/checkpoints'
BENCHMARK_DIR   = '/content/drive/MyDrive/ExamensArbete/benchmark_results'

print('Config:')
print(f'  3B model       : {MODEL_3B}')
print(f'  1.5B model       : {MODEL_1_5B}')
print(f'  Data fractions : {[int(f*100) for f in DATA_FRACTIONS]}%')
print(f'  Checkpoint dir : {DRIVE_SAVE_DIR}')

Config:
  3B model       : unsloth/Qwen2.5-3B-Instruct-bnb-4bit
  1.5B model       : unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit
  Data fractions : [2, 5, 10]%
  Checkpoint dir : /content/drive/MyDrive/ExamensArbete/checkpoints


## Test Run (optional)

Run the cell below **instead of Cells 5–8** to do a fast end-to-end check of the whole pipeline.
It trains the 3B model on 1% of the data (≈74 examples, 1 epoch) then runs inference on 20 questions.
Total time on an L4 should be under 15 minutes.

If this cell completes without errors the full pipeline is safe to run.

In [ ]:
# Test Cell — end-to-end pipeline check
# Trains 3B on 1% data then runs a 20-question mini benchmark.
# Run Cells 1-4 first so that dependencies are installed and config variables are set.

import gc
import os
import torch
from grpo_trainer import run_grpo
from benchmarker import (
    load_benchmark_questions,
    _load_trained_model,
    _run_nonagentic,
    _format_trained_prompt,
    _compute_metrics,
)
from utils import extract_tagged_answer

TEST_FRACTION    = 0.01   # ~74 training examples
TEST_CHECKPOINT  = f'{DRIVE_SAVE_DIR}/grpo_7b_1pct_best'
TEST_N_QUESTIONS = 20

# ── Step 1: Train ──────────────────────────────────────────────────────────────
print('=' * 60)
print('STEP 1: Training 3B on 1% data (1 epoch max)')
print('=' * 60)

if os.path.exists(TEST_CHECKPOINT):
    print(f'Checkpoint already exists at {TEST_CHECKPOINT} — skipping training.')
else:
    model, tokenizer, best_acc = run_grpo(
        model_name=MODEL_3B,
        data_fraction=TEST_FRACTION,
        save_dir=DRIVE_SAVE_DIR,
        warmup_path=WARMUP_PATH,
        max_epochs=1,   # single epoch keeps the test fast
        seed=SEED,
    )
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\nTraining complete. Best val accuracy: {best_acc:.4f}')

# ── Step 2: Mini benchmark (non-agentic, 20 questions) ─────────────────────────
print()
print('=' * 60)
print(f'STEP 2: Mini benchmark ({TEST_N_QUESTIONS} questions, 3B no agent)')
print('=' * 60)

questions, solutions = load_benchmark_questions(TEST_N_QUESTIONS, seed=SEED)
model, tokenizer = _load_trained_model(TEST_CHECKPOINT)
results = _run_nonagentic(
    model, tokenizer, questions, solutions,
    _format_trained_prompt, extract_tagged_answer,
)
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

metrics = _compute_metrics(results)
print(f'\nTest results ({TEST_N_QUESTIONS} questions):')
print(f'  Accuracy   : {metrics["accuracy"]:.2%}  ({metrics["correct"]}/{metrics["total"]})')
print(f'  Avg tokens : {metrics["avg_tokens"]:.0f}')
print()
print('✓ Pipeline completed without errors.')
print('  Note: accuracy will be low at 1% data — that is expected.')
print('  A non-zero accuracy (> 0) means the reward signal is working.')

Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
STEP 1: Training 7B on 1% data (1 epoch max)
==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Tool

Unsloth 2026.5.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model: 7b  |  Training examples: 74 (1%)
Validation examples: 100


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/25 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


...

SFT warmup complete -> /content/drive/MyDrive/ExamensArbete/checkpoints/grpo_7b_1pct_warmup


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 74 | Num Epochs = 1 | Total steps = 74
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / compute_reward / mean,rewards / compute_reward / std
10,0.001900,1.170000,0.536895,100.500000,77.200000,125.400000,0.000000,100.500000,77.200000,125.400000,0.047683,1.170000,0.536895
20,0.002200,1.408333,0.449431,73.425000,61.500000,88.200000,0.000000,73.425000,61.500000,88.200000,0.054524,1.408333,0.449431
30,0.001400,1.270000,0.375784,82.775000,66.600000,104.200000,0.000000,82.775000,66.600000,104.200000,0.035790,1.270000,0.375784
40,0.001800,1.170833,0.605309,97.400000,76.300000,120.500000,0.000000,97.400000,76.300000,120.500000,0.044372,1.170833,0.605309
50,0.001700,1.241667,0.484984,79.375000,64.200000,99.400000,0.000000,79.375000,64.200000,99.400000,0.043700,1.241667,0.484984
60,0.002600,1.208333,0.822042,77.850000,63.200000,94.000000,0.000000,77.850000,63.200000,94.000000,0.063802,1.208333,0.822042
70,0.001900,1.400000,0.824866,82.600000,67.600000,99.500000,0.000000,82.600000,67.600000,99.500000,0.046988,1.400000,0.824866


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
[Epoch 1] Val accuracy: 0.2800  Best: 0.0000
  -> New best saved to /content/drive/MyDrive/ExamensArbete/checkpoints/grpo_7b_1pct_best
  Metrics saved: /content/drive/MyDrive/ExamensArbete/checkpoints/grpo_7b_1pct_metrics.json
  Plot saved: /content/drive/MyDrive/ExamensArbete/checkpoints/grpo_7b_1pct_metrics.png
GRPO complete. Best val accuracy: 0.2800
Best checkpoint: /content/drive/MyDrive/ExamensArbete/checkpoints/grpo_7b_1pct_best

Training complete. Best val accuracy: 0.2800

STEP 2: Mini benchmark (20 questions, 7B no agent)
==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/u

## Training

Cells 5 and 6 each run 3 independent GRPO training runs (one per data fraction).
Each run: SFT warmup (1 epoch) → GRPO with early stopping (max 3 epochs).
Checkpoints are saved to Drive after every epoch — safe to resume if the session disconnects.
If a `_best` checkpoint already exists for a run, that run is skipped automatically.

In [7]:
import os
import time
import subprocess

RUNS = [
    (MODEL_3B, 0.025), (MODEL_3B, 0.05), (MODEL_3B, 0.10),
]

for model_name, frac in RUNS:
    size_tag = "3b" if "3B" in model_name else "1.5b"
    pct = frac * 100
    pct_str = str(pct).replace('.', '_') if pct % 1 == 0.5 else str(int(pct))
    expected = f"{DRIVE_SAVE_DIR}/grpo_{size_tag}_{pct_str}pct_best"
    
    if os.path.exists(expected):
        print(f"SKIP  {size_tag} {pct_str}% — checkpoint finns redan")
        continue

    print(f"\n{'='*60}\nSTART {size_tag} {pct_str}%\n{'='*60}", flush=True)

    # GPU-status innan start — visar om minnet redan är upptaget.
    print("--- nvidia-smi (före start) ---", flush=True)
    subprocess.run(
        ["nvidia-smi",
         "--query-gpu=memory.used,memory.total",
         "--format=csv,noheader"],
    )

    # Logga till Drive så att loggen överlever även om kerneln/sessionen dör.
    log_path = f"{DRIVE_SAVE_DIR}/grpo_{size_tag}_{pct_str}pct_run.log"
    with open(log_path, "w") as logf:
        proc = subprocess.Popen(
            ["python", "start_grpo_sub_process.py",
             "--model", model_name,
             "--fraction", str(frac),
             "--save_dir", DRIVE_SAVE_DIR,
             "--max_epochs", str(MAX_EPOCHS),
             "--seed", str(SEED)],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,   # fånga även tracebacks i samma ström
            text=True,
            bufsize=1,                  # radbuffrat
        )
        for line in proc.stdout:        # strömmar rad för rad
            print(line, end="", flush=True)   # syns live i cellen
            logf.write(line)
            logf.flush()                # ligger på Drive direkt
        proc.wait()

    returncode = proc.returncode
    print(f"\nEND   {size_tag} {pct_str}%  (returncode {returncode})", flush=True)
    print(f"  Logg sparad: {log_path}", flush=True)

    if returncode != 0:
        print(f"  VARNING: körningen avslutades med fel — fortsätter till nästa.", flush=True)
        # Tolkning av returkoden:
        #   -9 / 137  = OOM-kill (slut på GPU- eller RAM-minne)
        #   annat     = se tracebacken ovan / i loggfilen
        if returncode in (-9, 137):
            print("  -> Trolig OOM-kill. Minska batch/generations i GRPOConfig.", flush=True)

    # Ge GPU-minnet tid att frigöras helt innan nästa subprocess startar.
    time.sleep(15)

print("\nAlla körningar klara (eller överhoppade).")


START 3b 2_5%
--- nvidia-smi (före start) ---
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\ 

In [8]:
import os
import time
import subprocess

RUNS = [
    #(MODEL_1_5B, 0.025),
      (MODEL_1_5B, 0.05), (MODEL_1_5B, 0.10),
]

for model_name, frac in RUNS:
    size_tag = "3b" if "3B" in model_name else "1_5b"
    pct = frac * 100
    pct_str = str(pct_str).replace('.', '_') if pct % 1 == 0.5 else str(int(pct))
    expected = f"{DRIVE_SAVE_DIR}/grpo_{size_tag}_{pct_str}pct_best"

    if os.path.exists(expected):
        print(f"SKIP  {size_tag} {pct_str}% — checkpoint finns redan")
        continue

    print(f"\n{'='*60}\nSTART {size_tag} {pct_str}%\n{'='*60}", flush=True)

    # GPU-status innan start — visar om minnet redan är upptaget.
    print("--- nvidia-smi (före start) ---", flush=True)
    subprocess.run(
        ["nvidia-smi",
         "--query-gpu=memory.used,memory.total",
         "--format=csv,noheader"],
    )

    # Logga till Drive så att loggen överlever även om kerneln/sessionen dör.
    log_path = f"{DRIVE_SAVE_DIR}/grpo_{size_tag}_{pct_str}pct_run.log"
    with open(log_path, "w") as logf:
        proc = subprocess.Popen(
            ["python", "-u", "start_grpo_sub_process.py",
             "--model", model_name,
             "--fraction", str(frac),
             "--save_dir", DRIVE_SAVE_DIR,
             "--max_epochs", str(MAX_EPOCHS),
             "--seed", str(SEED)],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,   # fånga även tracebacks i samma ström
            text=True,
            bufsize=1,                  # radbuffrat
        )
        for line in proc.stdout:        # strömmar rad för rad
            print(line, end="", flush=True)   # syns live i cellen
            logf.write(line)
            logf.flush()                # ligger på Drive direkt
        proc.wait()

    returncode = proc.returncode
    print(f"\nEND   {size_tag} {pct_str}%  (returncode {returncode})", flush=True)
    print(f"  Logg sparad: {log_path}", flush=True)

    if returncode != 0:
        print(f"  VARNING: körningen avslutades med fel — fortsätter till nästa.", flush=True)
        # Tolkning av returkoden:
        #   -9 / 137  = OOM-kill (slut på GPU- eller RAM-minne)
        #   annat     = se tracebacken ovan / i loggfilen
        if returncode in (-9, 137):
            print("  -> Trolig OOM-kill. Minska batch/generations i GRPOConfig.", flush=True)

    # Ge GPU-minnet tid att frigöras helt innan nästa subprocess startar.
    time.sleep(15)

print("\nAlla körningar klara (eller överhoppade).")

SKIP  1_5b 5% — checkpoint finns redan

START 1_5b 10%
--- nvidia-smi (före start) ---
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
==((====))==  Unsloth 2026.5.8: Fast Qwen2 patching. Tr

: 

## Benchmark

Uses the 40% checkpoint for Groups 1-4. Group 5 is the zero-shot Qwen2.5-14B-Instruct baseline.
500 questions from the official GSM8K test split (fixed seed = 42).

In [ ]:
# Cell 7 — Verify all required checkpoints exist
import os

to_check = {
    '3B 2.5%': f'{DRIVE_SAVE_DIR}/grpo_3b_2_5pct_best',
    '3B 5%': f'{DRIVE_SAVE_DIR}/grpo_3b_5pct_best',
    '3B 10%': f'{DRIVE_SAVE_DIR}/grpo_3b_10pct_best',
    '1.5B 2.5%': f'{DRIVE_SAVE_DIR}/grpo_1_5b_2_5pct_best',
    '1.5B 5%': f'{DRIVE_SAVE_DIR}/grpo_1_5b_5pct_best',
    '1.5B 1+%': f'{DRIVE_SAVE_DIR}/grpo_1_5b_10pct_best',
}

all_ok = True
for label, path in to_check.items():
    status = 'OK     ' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}]  {label}  ->  {path}')
    if 'MISSING' in status:
        all_ok = False

print()
if all_ok:
    print('All checkpoints present. Ready to run benchmark.')
else:
    print('Some checkpoints are missing. Complete training before running the benchmark.')

In [ ]:
# Cell 7a — Configuration & 2-Question Sanity Test Run
import os
from benchmarker import run_h0_benchmark, run_h1_benchmark, run_h2_benchmark

# 1. Define Paths to Checkpoints (Adjust these to your actual Drive structure)
CHECKPOINTS_H1 = {
    "1_5b_2pct":  f"{DRIVE_SAVE_DIR}/grpo_1_5b_2pct_best",
    "1_5b_5pct":  f"{DRIVE_SAVE_DIR}/grpo_1_5b_5pct_best",
    "1_5b_10pct": f"{DRIVE_SAVE_DIR}/grpo_1_5b_10pct_best",
    "3b_2pct":    f"{DRIVE_SAVE_DIR}/grpo_3b_2pct_best",
    "3b_5pct":    f"{DRIVE_SAVE_DIR}/grpo_3b_5pct_best",
    "3b_10pct":   f"{DRIVE_SAVE_DIR}/grpo_3b_10pct_best",
}

# Default choice for Agentic Compute loop evaluation (H2)
AGENT_CHECKPOINT_H2 = CHECKPOINTS_H1["3b_10pct"]

# Setup evaluation directory
TEST_BENCHMARK_DIR = os.path.join(BENCHMARK_DIR, "test_run")

print("=" * 70)
print("LAUNCHING 2-QUESTION SANITY CHECK FOR ALL EVALUATION HYPOTHESES")
print("=" * 70)

# Run H0 Baseline Quick Check
print("\n--- Testing Hypothesis 0 (Untrained Models) ---")
h0_test_metrics = run_h0_benchmark(
    save_dir=TEST_BENCHMARK_DIR,
    n_questions=2,
    seed=SEED
)

# Run H1 Scale & Data Efficiency Quick Check
print("\n--- Testing Hypothesis 1 (GRPO No-Agent Checkpoints) ---")
h1_test_metrics = run_h1_benchmark(
    checkpoints=CHECKPOINTS_H1,
    save_dir=TEST_BENCHMARK_DIR,
    n_questions=2,
    seed=SEED
)

# Run H2 Agentic vs SOTA Quick Check
print("\n--- Testing Hypothesis 2 (Agentic Process + Math Baseline) ---")
h2_test_metrics = run_h2_benchmark(
    agent_checkpoint=AGENT_CHECKPOINT_H2,
    save_dir=TEST_BENCHMARK_DIR,
    n_questions=2,
    seed=SEED,
    skip_baseline=False
)

print("\n" + "=" * 70)
print("SUCCESS: 2-Question test runs completed without sub-process failures.")
print("=" * 70)

In [ ]:
# Cell 9 — Display results table and plot
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f'{BENCHMARK_DIR}/benchmark_summary.csv')
df['accuracy_%']         = (df['accuracy'] * 100).round(2)
df['avg_tokens']         = df['avg_tokens'].round(1)
df['tokens_per_correct'] = df['tokens_per_correct'].round(1)

display(df[['group_name', 'accuracy_%', 'avg_tokens', 'tokens_per_correct', 'correct', 'total']])

print()
display(Image(filename=f'{BENCHMARK_DIR}/benchmark_plot.png'))